In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_exposure_request_profits(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        # "ExposureRequestId string, SourceSystem string, PortfolioKey int, PortfolioName string,"
        # "IngestionTimestamp timestamp, BatchId string, ExposureRequestKey string, ExposureKey string, RequestSN int,"
        # "RequestTypeCode string, RequestType string, RequestStatusCode string, RequestStatus string, ExposureStatusCode string,"
        # "ExposureStatus string, InterestAmount decimal(19,2), AccruedInterestAmount decimal(19,2), AccruedPenaltyInterestAmount decimal(19,2),"
        # "AccruedPublicInterestAmount decimal(19,2), ExpenseAmount decimal(19,2), CommisionAmount decimal(19,2), UnrealizedInterestAmount decimal(19,2),"
        # "PenaltyInterestAmount decimal(19,2), SubsidyAmount decimal(19,2), OverdueAccruedAmount decimal(19,2), UnrealizedPenaltyInterestAmount decimal(19,2),"
        # "UnrealizedPublicInterestAmount decimal(19,2), CapitalBalance decimal(19,2), RequestCreatedDate date, RequestExpiryDate date, BusinessDate Date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExposureRequest"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "profits",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # =================  Portfolio ==========================#

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== loan_request ==================

        loan_request_extra_filters = None

        if loan_request_extra_filters is not None:
            loan_request_extra_filters.append(date_filter)
            loan_request_final_filters = " AND ".join(loan_request_extra_filters)
        else:
            loan_request_final_filters = date_filter

        loan_request_selected_fields = [
            "REQUEST_TYPE",
            "REQUEST_LOAN_STS",
            "unit_code",
            "REQUEST_SN",
            "ACCOUNT_NUMBER",
            "REQUEST_STS",
            "RQ_RL_NRM_INT_BAL",
            "RQ_ACR_NRM_INT_BAL",
            "RQ_ACR_PNL_INT_BAL",
            "RQ_ACR_PUB_INT_AMN",
            "RQ_EXP_BAL",
            "RQ_COM_BAL",
            "RQ_URL_NRM_INT_BAL",
            "RQ_RL_PNL_INT_BAL",
            "RQ_SUBSIDY_BAL",
            "OV_ACCRUAL_AMN",
            "RQ_URL_PNL_INT_BAL",
            "RQ_URL_PUB_INT_AMN",
            "RQ_CAPITAL_BAL",
            "RQ_CREATION_DT",
            "RQ_EXPIRE_DT",
            "businessdate",
        ]

        loan_request = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "profits_loan_request",
            loan_request_final_filters,
            loan_request_selected_fields,
        )

        # HashKeys
        loan_request_ExposureRequestId_primary_key_fields = [
            loan_request.REQUEST_TYPE,
            loan_request.REQUEST_LOAN_STS,
            loan_request.REQUEST_SN,
            loan_request.ACCOUNT_NUMBER,
            lit("profits"),
        ]

        loan_request = (
            loan_request.withColumn(
                "ExposureRequestId",
                sha2(
                    concat_ws(";", *loan_request_ExposureRequestId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "ExposureRequestKey",
                concat_ws(
                    ";",
                    loan_request.REQUEST_TYPE,
                    loan_request.REQUEST_LOAN_STS,
                    loan_request.REQUEST_SN,
                    loan_request.ACCOUNT_NUMBER,
                ),
            )
            .withColumn(
                "ExposureStatus",
                when(loan_request.REQUEST_LOAN_STS == "1", "Normal")
                .when(
                    loan_request.REQUEST_LOAN_STS == "2",
                    "Overdue",
                )
                .when(
                    loan_request.REQUEST_LOAN_STS == "3",
                    "Define Delay",
                )
                .when(loan_request.REQUEST_LOAN_STS == "4", "Writen Off")
                .otherwise(None),
            )
            .withColumn(
                "RequestType",
                when(loan_request.REQUEST_TYPE == "1", "Installment")
                .when(
                    loan_request.REQUEST_TYPE == "2",
                    "Interest Request",
                )
                .when(
                    loan_request.REQUEST_TYPE == "3",
                    "Loan Request",
                )
                .when(loan_request.REQUEST_TYPE == "4", "ExpenseRequest")
                .when(loan_request.REQUEST_TYPE == "5", "CapitalRequest")
                .otherwise(None),
            )
        )

        # ====================== JOINS ===================================#
        loan_request_portfolio_join_condition = (
            loan_request["unit_code"] == portfolio["unit_code_profits"]
        )

        result_df = loan_request.join(
            portfolio, loan_request_portfolio_join_condition, "left"
        )

        # ================== Selection of Final Columns ==================

        result_df_profits = result_df.select(
            loan_request.ExposureRequestId.alias("ExposureRequestId"),
            lit("profits").alias("SourceSystem"),
            loan_request.unit_code.alias("PortfolioKey"),
            portfolio.PortfolioName.alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            loan_request.ExposureRequestKey.alias("ExposureRequestKey"),
            loan_request.ACCOUNT_NUMBER.alias("ExposureKey"),
            loan_request.REQUEST_SN.alias("RequestSN"),
            loan_request.REQUEST_TYPE.alias("RequestTypeCode"),
            loan_request.RequestType.alias("RequestType"),
            loan_request.REQUEST_STS.alias("RequestStatusCode"),
            loan_request.REQUEST_STS.alias("RequestStatus"),
            loan_request.REQUEST_LOAN_STS.alias("ExposureStatusCode"),
            loan_request.ExposureStatus.cast("string").alias("ExposureStatus"),
            loan_request.RQ_RL_NRM_INT_BAL.cast("decimal(19,2)").alias(
                "InterestAmount"
            ),
            loan_request.RQ_ACR_NRM_INT_BAL.cast("decimal(19,2)").alias(
                "AccruedInterestAmount"
            ),
            loan_request.RQ_ACR_PNL_INT_BAL.cast("decimal(19,2)").alias(
                "AccruedPenaltyInterestAmount"
            ),
            loan_request.RQ_ACR_PUB_INT_AMN.cast("decimal(19,2)").alias(
                "AccruedPublicInterestAmount"
            ),
            loan_request.RQ_EXP_BAL.cast("decimal(19,2)").alias("ExpenseAmount"),
            loan_request.RQ_COM_BAL.cast("decimal(19,2)").alias("CommisionAmount"),
            loan_request.RQ_URL_NRM_INT_BAL.cast("decimal(19,2)").alias(
                "UnrealizedInterestAmount"
            ),
            loan_request.RQ_RL_PNL_INT_BAL.cast("decimal(19,2)").alias(
                "PenaltyInterestAmount"
            ),
            loan_request.RQ_SUBSIDY_BAL.cast("decimal(19,2)").alias("SubsidyAmount"),
            loan_request.OV_ACCRUAL_AMN.cast("decimal(19,2)").alias(
                "OverdueAccruedAmount"
            ),
            loan_request.RQ_URL_PNL_INT_BAL.cast("decimal(19,2)").alias(
                "UnrealizedPenaltyInterestAmount"
            ),
            loan_request.RQ_URL_PUB_INT_AMN.cast("decimal(19,2)").alias(
                "UnrealizedPublicInterestAmount"
            ),
            loan_request.RQ_CAPITAL_BAL.cast("decimal(19,2)").alias("CapitalBalance"),
            loan_request.RQ_CREATION_DT.alias("RequestCreatedDate"),
            loan_request.RQ_EXPIRE_DT.alias("RequestExpiryDate"),
            loan_request.businessdate.alias("BusinessDate"),
        )

        result_df_profits.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_exposure_request_profits(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)